# Step 1: lock in the RAMP baseline (official code)

**Once:**
1. Kaggle *Add-ons → Secrets*: `HF_TOKEN` = Hugging Face **write** token, attached to this notebook.
2. On your tablet or phone, install the **ntfy** app and subscribe to the topic in `NTFY_TOPIC` below. Every stage sends you a push when it starts and when it ends: **OK**, **FAILED** (with the error), or **PAUSED** (time budget reached, so run the notebook again).

**Each session:** GPU **T4 x2**, Internet **on**, then **Save Version → Save & Run All**. The notebook stops at the first failed stage, and every stage's full log is uploaded to Hugging Face under `logs/`.

In [ ]:
import os
os.environ['HF_REPO'] = 'matokebryan/aat-checkpoints'
os.environ['NTFY_TOPIC'] = 'aat-matoke-7q4r9x'   # subscribe to this in the ntfy app
os.environ['TIME_BUDGET_H'] = '10.5'             # training stops cleanly after this and pushes its checkpoint
MODE = 'scratch'                                 # 'scratch' = thesis baseline (multi-session), 'finetune' = cheap cross-check
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
%cd /kaggle/working
!git clone -q -b matt/funny-planck-3j7md5 https://github.com/mattobryan/AAT.git 2>/dev/null || (cd AAT && git pull -q)
%cd /kaggle/working/AAT

def stage(name, cmd):
    """Run a stage, stream its output, push a phone notification, and stop the notebook if it fails."""
    get_ipython().system(f'python scripts/stage.py {name} "{cmd}"')
    if _exit_code != 0:
        raise SystemExit(f'stage {name} failed (exit {_exit_code}): see the notification or logs/{name}.txt on Hugging Face')

stage('setup', 'bash scripts/ramp_official.sh setup')

## Sanity check: the official pretrained ℓ∞ model must give ≈ 83.7 / 48.1 / 59.8 / 7.7 / 38.5

In [ ]:
if not os.path.exists('runs_official/pretr_linf/eval_autoattack.json'):
    stage('sanity', 'bash scripts/ramp_official.sh pretr 0 && python scripts/compare_targets.py --runs runs_official --map pretr_linf=pretr_linf')

## A. Thesis baseline: RAMP from scratch (λ=5, 80 epochs, GP), seeds 0 and 1 in parallel
Each session trains until the time budget, pushes its checkpoint, and sends **PAUSED**. Run the notebook again, and it resumes. When both seeds finish, the evaluation runs automatically and sends the comparison with the thesis Table 7.1.

In [ ]:
def finished(run):
    return os.path.exists(f'external/ramp/trained_models/{run}/log_eval_final.txt')

if MODE == 'scratch':
    stage('ramp_scratch_train', 'bash scripts/ramp_official.sh train2 ramp_scratch 0 1')
    if all(finished(f'ramp_scratch_l5_s{s}') for s in (0, 1)):
        stage('ramp_scratch_eval', 'bash scripts/ramp_official.sh eval2 ramp_scratch 0 1 && '
              'python scripts/compare_targets.py --runs runs_official --table thesis_table7_1 --map ramp_scratch_official=ramp; '
              'python scripts/compare_targets.py --runs runs_official --table scratch_table3 --map ramp_scratch_official=ramp_l5')
    else:
        print('Training paused at the time budget. Run the notebook again to resume.')

## B. Cross-check: RAMP fine-tuning (paper Table 24), 5 seeds, about 1 GPU-hour each

In [ ]:
if MODE == 'finetune':
    stage('ramp_ft_train', 'bash scripts/ramp_official.sh train2 ramp \\"0 2 4\\" \\"1 3\\"')
    stage('ramp_ft_eval', 'bash scripts/ramp_official.sh eval2 ramp \\"0 2 4\\" \\"1 3\\" && '
          'python scripts/compare_targets.py --runs runs_official --map ramp_official=ramp_l1.5 pretr_linf=pretr_linf')